# 🔬 CellOneFormer: Evaluation, Hyperparameter Tuning & Test Benchmark
### Pipeline d'évaluation et d'inférence basé sur le *Multi-modality Cell Segmentation Challenge (Nature Methods 2024)*

Ce notebook permet de :
1. **Charger vos poids entraînés** (`oneformer_cells_best.pt`) directement depuis Google Drive.
2. **Optimiser les hyperparamètres de post-traitement (*Tuning*)** sur le jeu `tuning/` : seuil de confiance de détection (`score_threshold`), seuil de fusion des coutures (*Soft-IoU stitching*), et taille minimale d'aire cellulaire (`min_cell_area`).
3. **Calculer les métriques officielles du challenge** : $\text{AP}_{50}$, $\text{AP}_{75}$, $\text{mAP}_{[50:95]}$ et score F1.
4. **Générer les prédictions finales (*Testing*)** sur les images pleine résolution de `testing/`.
5. **Visualiser les galeries de segmentation d'instances** (comparaison côte-à-côte haute résolution).
6. **Exporter les masques de soumission** automatiquement vers votre Google Drive.

## 1. Vérification du GPU & Montage de Google Drive
Assurez-vous d'être connecté à un GPU (T4 ou A100) : **Exécution > Modifier le type d'exécution > GPU**.

In [ ]:
!nvidia-smi

from google.colab import drive
import os

drive.mount('/content/drive')
print("✅ Google Drive connecté avec succès.")

## 2. Configuration de l'environnement & Dépendances
Clonage ou mise à jour du dépôt et installation des bibliothèques nécessaires.

In [ ]:
import os

REPO_DIR = "/content/CellOneFormer"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/BaptFontaine/CellOneFormer.git "{REPO_DIR}"
else:
    !cd "{REPO_DIR}" && git pull

%cd /content/CellOneFormer
%env PYTHONPATH=/content/CellOneFormer

# Optimisation de la mémoire GPU
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Installation des dépendances
!pip install -q -r requirements.txt
print("✅ Environnement CellOneFormer prêt !")

## 3. Chargement du Modèle & Vérification des Poids Entraînés
Récupération automatique du checkpoint `oneformer_cells_best.pt` sauvegardé sur votre Google Drive.

In [ ]:
import os
import shutil

DRIVE_CHECKPOINT = "/content/drive/MyDrive/cell_oneformer_checkpoints/oneformer_cells_best.pt"
LOCAL_CHECKPOINT = "/content/CellOneFormer/checkpoints/oneformer_cells_best.pt"

os.makedirs("/content/CellOneFormer/checkpoints", exist_ok=True)

if os.path.exists(DRIVE_CHECKPOINT):
    if not os.path.exists(LOCAL_CHECKPOINT):
        shutil.copy(DRIVE_CHECKPOINT, LOCAL_CHECKPOINT)
    size_mb = os.path.getsize(LOCAL_CHECKPOINT) / (1024 * 1024)
    print(f"✅ Checkpoint chargé avec succès : {LOCAL_CHECKPOINT} ({size_mb:.1f} Mo)")
elif os.path.exists(LOCAL_CHECKPOINT):
    size_mb = os.path.getsize(LOCAL_CHECKPOINT) / (1024 * 1024)
    print(f"✅ Checkpoint local trouvé : {LOCAL_CHECKPOINT} ({size_mb:.1f} Mo)")
else:
    print(f"⚠️ Attention : Checkpoint introuvable à {DRIVE_CHECKPOINT}.")
    print("Vérifiez le chemin de votre dossier dans Google Drive.")

## 4. Préparation & Détection des Données (`tuning` et `testing`)
Copie des images de validation et de test depuis Google Drive vers le SSD local de Colab pour maximiser les performances d'inférence.

In [ ]:
import os
import glob

DRIVE_DATASET = "/content/drive/MyDrive/DatasetCellSeg"
LOCAL_DATASET = "/content/CellOneFormer/data/raw/DatasetCellSeg"
os.makedirs(LOCAL_DATASET, exist_ok=True)

if os.path.exists(DRIVE_DATASET):
    print(f"📁 Synchronisation depuis {DRIVE_DATASET} vers {LOCAL_DATASET}...")
    !cp -ru "{DRIVE_DATASET}"/* "{LOCAL_DATASET}/"

# Recherche et inventaire des images de tuning (validation) et testing
def find_images_and_masks(split_name):
    split_dir = os.path.join(LOCAL_DATASET, split_name)
    if not os.path.exists(split_dir):
        return [], []
    
    all_files = glob.glob(f"{split_dir}/**/*.*", recursive=True)
    imgs = [f for f in all_files if f.lower().endswith(('.png', '.tif', '.tiff', '.jpg', '.jpeg')) and not any(k in f.lower() for k in ['mask', 'label'])]
    masks = [f for f in all_files if f.lower().endswith(('.png', '.tif', '.tiff', '.jpg', '.jpeg')) and any(k in f.lower() for k in ['mask', 'label'])]
    return sorted(imgs), sorted(masks)

tuning_imgs, tuning_masks = find_images_and_masks("tuning")
test_imgs, _ = find_images_and_masks("testing")

# Fallback si tuning est vide : utiliser un sous-ensemble d'images avec masques pour calibrer
if len(tuning_imgs) == 0:
    train_imgs, train_masks = find_images_and_masks("training")
    if len(train_imgs) > 0:
        print("ℹ️ Pas d'images séparées dans tuning/. Utilisation d'un échantillon annoté pour le tuning des seuils.")
        tuning_imgs, tuning_masks = train_imgs[:3], train_masks[:3]

print(f"🔬 Images disponibles pour le Tuning (validation) : {len(tuning_imgs)}")
print(f"🚀 Images disponibles pour le Test final         : {len(test_imgs)}")

## 5. Hyperparameter Tuning sur le Split de Validation
Cette étape optimise les paramètres de post-traitement du modèle pour maximiser le score $\text{mAP}_{[50:95]}$ :
- `score_threshold` : Seuil de probabilité minimale pour conserver une cellule détectée.
- `iou_threshold_stitch` : Seuil Soft-IoU de déduplication des cellules chevauchant la couture de deux tuiles.
- `min_cell_area` : Filtrage des détections parasites de moins de $N$ pixels.

In [ ]:
import yaml
import numpy as np
from PIL import Image
import torch

from src.dataset.tiler import SlidingWindowTiler, load_microscopy_image, load_instance_mask
from src.inference.stitcher import TileStitcher
from src.models.oneformer_wrapper import CellOneFormerModule
from src.utils.metrics import compute_challenge_benchmark_metrics

# Grille d'hyperparamètres à évaluer
SCORE_THRESHOLDS = [0.3, 0.4, 0.5, 0.6]
STITCH_IOUS = [0.4, 0.5, 0.6]
MIN_AREAS = [15, 30]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
with open("configs/config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

print(f"Chargement du modèle sur {device}...")
model_module = CellOneFormerModule(
    pretrained_model_name=cfg["model"]["pretrained_model_name"],
    num_classes=cfg["model"]["num_classes"],
    num_queries=cfg["model"]["num_queries"],
)
if os.path.exists(LOCAL_CHECKPOINT):
    model_module.load_state_dict(torch.load(LOCAL_CHECKPOINT, map_location=device))
model_module.to(device)
model_module.eval()

# Fonction d'évaluation d'un ensemble de paramètres
def evaluate_tuning_config(score_th, stitch_iou, min_area, max_eval_samples=2):
    tiler = SlidingWindowTiler(tile_size=512, tile_overlap=64)
    all_metrics = []

    for img_p in tuning_imgs[:max_eval_samples]:
        stem = os.path.splitext(os.path.basename(img_p))[0]
        matching_masks = [m for m in tuning_masks if stem in os.path.basename(m)]
        if not matching_masks:
            continue
        
        image = load_microscopy_image(img_p)
        gt_mask = load_instance_mask(matching_masks[0])
        gt_ids = np.unique(gt_mask)
        gt_instances = [gt_mask == gid for gid in gt_ids if gid > 0]
        if not gt_instances:
            continue
        gt_instances = np.array(gt_instances)

        h, w = image.shape[:2]
        tiles = tiler.tile_image_and_mask(image)
        tile_results = []

        with torch.no_grad():
            for tile in tiles:
                pil_img = Image.fromarray(tile["image"])
                inputs = model_module.processor(images=pil_img, task_inputs=["instance"], return_tensors="pt")
                device_inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}
                outputs = model_module(**device_inputs)

                post = model_module.post_process_instances(
                    outputs,
                    target_sizes=[(512, 512)],
                    threshold=score_th,
                )
                t_masks = []
                t_scores = []
                if len(post) > 0 and "segmentation" in post[0]:
                    imap = post[0]["segmentation"].cpu().numpy()
                    for uid in np.unique(imap):
                        if uid > 0:
                            t_masks.append(imap == uid)
                            t_scores.append(1.0)
                tile_results.append({"masks": t_masks, "scores": t_scores, "coords": tile["coords"]})

        stitcher = TileStitcher(iou_threshold=stitch_iou, min_area=min_area)
        _, kept_masks = stitcher.stitch_tile_predictions(h, w, tile_results)
        metrics = compute_challenge_benchmark_metrics(kept_masks, gt_instances)
        all_metrics.append(metrics)

    if not all_metrics:
        return {"AP@50": 0.0, "AP@75": 0.0, "mAP@[50:95]": 0.0, "F1": 0.0}
    
    return {
        "AP@50": np.mean([m.get("AP@50", 0.0) for m in all_metrics]),
        "AP@75": np.mean([m.get("AP@75", 0.0) for m in all_metrics]),
        "mAP@[50:95]": np.mean([m.get("mAP@[50:95]", 0.0) for m in all_metrics]),
        "F1": np.mean([m.get("mean_f1", 0.0) for m in all_metrics]),
    }

print("🔍 Lancement de la recherche d'hyperparamètres (Tuning Grid)...")
tuning_results = []
for s_th in SCORE_THRESHOLDS:
    for st_iou in STITCH_IOUS:
        res = evaluate_tuning_config(score_th=s_th, stitch_iou=st_iou, min_area=15)
        tuning_results.append({
            "score_threshold": s_th,
            "stitch_iou": st_iou,
            "min_area": 15,
            **res
        })

tuning_results.sort(key=lambda x: x.get("mAP@[50:95]", 0.0), reverse=True)
print("\n🏆 Résultats du Tuning (classés par mAP@[50:95]) :\n")
print("=" * 75)
print(f" {'Score Th':<10} | {'Stitch IoU':<12} | {'AP@50':<10} | {'AP@75':<10} | {'mAP@[50:95]':<12} | {'F1':<8}")
print("-" * 75)
for r in tuning_results:
    print(f" {r['score_threshold']:<10.2f} | {r['stitch_iou']:<12.2f} | {r['AP@50']:<10.4f} | {r['AP@75']:<10.4f} | {r['mAP@[50:95]']:<12.4f} | {r['F1']:<8.4f}")
print("=" * 75)

best_config = tuning_results[0]
print(f"\n⭐ Meilleure configuration trouvée : score_threshold={best_config['score_threshold']}, stitch_iou={best_config['stitch_iou']}")

## 6. Inférence & Benchmark sur le Jeu de Test (`testing/`)
Applique les paramètres optimaux déterminés lors du Tuning pour générer les masques pleine résolution de l'ensemble des images de test.

In [ ]:
import os
import glob
from pathlib import Path

OUTPUT_PRED_DIR = "/content/CellOneFormer/predictions_test"
os.makedirs(OUTPUT_PRED_DIR, exist_ok=True)

best_score_th = best_config.get("score_threshold", 0.5)
best_stitch_iou = best_config.get("stitch_iou", 0.5)
best_min_area = int(best_config.get("min_area", 15))

# Cibles d'inférence (priorité à testing/, puis validation)
target_imgs = test_imgs if len(test_imgs) > 0 else tuning_imgs
print(f"🚀 Inférence sur {len(target_imgs)} image(s) pleine résolution avec paramètres optimaux :")
print(f"   - Score Threshold : {best_score_th}")
print(f"   - Soft-IoU Stitch : {best_stitch_iou}")
print(f"   - Min Cell Area   : {best_min_area} px\n")

for idx, img_path in enumerate(target_imgs, 1):
    stem = os.path.splitext(os.path.basename(img_path))[0]
    print(f"[{idx}/{len(target_imgs)}] Traitement de : {stem}...")
    
    # Recherche d'un masque de vérité terrain si présent
    mask_matches = [m for m in tuning_masks if stem in os.path.basename(m)]
    mask_flag = f'--mask "{mask_matches[0]}"' if mask_matches else ""
    
    !python src/inference/predict.py \
        --image "{img_path}" \
        {mask_flag} \
        --weights "{LOCAL_CHECKPOINT}" \
        --score-threshold {best_score_th} \
        --stitch-iou {best_stitch_iou} \
        --min-area {best_min_area} \
        --output-dir "{OUTPUT_PRED_DIR}"

print(f"\n✅ Inférence terminée ! Toutes les prédictions sont enregistrées dans {OUTPUT_PRED_DIR}.")

## 7. Galerie Visuelle des Résultats Haute Résolution
Affiche les images de microscopie originales et la segmentation d'instances reconstituée par CellOneFormer.

In [ ]:
import cv2
import matplotlib.pyplot as plt

overlays = sorted(glob.glob(f"{OUTPUT_PRED_DIR}/*_overlay.png"))
num_to_display = min(4, len(overlays))

if num_to_display > 0:
    fig, axes = plt.subplots(num_to_display, 2, figsize=(16, 7 * num_to_display))
    if num_to_display == 1:
        axes = np.expand_dims(axes, axis=0)
        
    for i in range(num_to_display):
        ov_p = overlays[i]
        stem = os.path.basename(ov_p).replace("_overlay.png", "")
        orig_candidates = [p for p in target_imgs if stem in os.path.basename(p)]
        
        if orig_candidates:
            orig_img = cv2.cvtColor(cv2.imread(orig_candidates[0]), cv2.COLOR_BGR2RGB)
            axes[i, 0].imshow(orig_img)
            axes[i, 0].set_title(f"Original : {stem}", fontsize=13)
            axes[i, 0].axis("off")
            
        ov_img = cv2.cvtColor(cv2.imread(ov_p), cv2.COLOR_BGR2RGB)
        axes[i, 1].imshow(ov_img)
        axes[i, 1].set_title(f"CellOneFormer Instances (Stitched) : {stem}", fontsize=13)
        axes[i, 1].axis("off")
        
    plt.tight_layout()
    plt.show()
else:
    print("Aucun overlay trouvé pour l'affichage.")

## 8. Exportation & Sauvegarde des Résultats sur Google Drive
Archive les masques prédits au format officiel (PNG 16-bit) et les prévisualisations directement sur votre Drive.

In [ ]:
import shutil
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
DRIVE_EXPORT_DIR = f"/content/drive/MyDrive/CellOneFormer_Results_{timestamp}"
os.makedirs(DRIVE_EXPORT_DIR, exist_ok=True)

# Copie des masques et visuels vers Google Drive
!cp -r "{OUTPUT_PRED_DIR}"/* "{DRIVE_EXPORT_DIR}/"

# Création d'une archive zip téléchargeable
zip_name = f"/content/CellOneFormer_predictions_{timestamp}"
shutil.make_archive(zip_name, 'zip', OUTPUT_PRED_DIR)
shutil.copy(f"{zip_name}.zip", DRIVE_EXPORT_DIR)

print(f"✅ Tous les résultats ont été exportés sur Google Drive dans :")
print(f"   📁 {DRIVE_EXPORT_DIR}")
print(f"   📦 Archive : {DRIVE_EXPORT_DIR}/CellOneFormer_predictions_{timestamp}.zip")